In [1]:
import pandas as pd
import numpy as np

In [2]:
df_path = "/Users/antonio/Desktop/PT_Tortugas/Data/Data_Joined.csv"
df_joined = pd.read_csv(df_path)

In [3]:
df_joined.info()

<class 'pandas.DataFrame'>
RangeIndex: 4882 entries, 0 to 4881
Data columns (total 61 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Unnamed: 0                                      4882 non-null   int64  
 1   AÑO                                             4882 non-null   int64  
 2   No. NIDO                                        4882 non-null   float64
 3   ESPECIE                                         4882 non-null   str    
 4   FECHA DE RECOLECTA                              4882 non-null   str    
 5   HORA DE AVISTAMIENTO                            4882 non-null   str    
 6   ESTACION                                        4882 non-null   float64
 7   ZONA                                            4882 non-null   str    
 8   SE OBSERVO NIDO                                 4882 non-null   float64
 9   SE OBSERVO TORTUGA                              4882

In [4]:
# --- Preparación ---
date_cols = [
"FECHA DE RECOLECTA",
"FECHA PROBABLE DE ECLOSION",
"FECHA DE EMERGENCIA",
"fecha_hora_recolecta",

]

for col in date_cols:
    df_joined[col] = pd.to_datetime(df_joined[col], errors="coerce")

/var/folders/yc/z556vm1x5l720dmsfkbcxn2r0000gp/T/ipykernel_45174/607715567.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_joined[col] = pd.to_datetime(df_joined[col], errors="coerce")


In [5]:
df_joined["time_local"].head()

0    2018-04-07 12:00:00
1    2018-05-15 13:00:00
2    2018-05-16 10:00:00
3    2018-05-19 10:00:00
4    2018-05-19 10:00:00
Name: time_local, dtype: str

In [6]:
# Función para extraer el promedio del clima durante la vida del nido
def get_incubation_weather(row, clima_df):
    import numpy as np
    mask = (clima_df['time_local'] >= row['fecha_inicio']) & \
           (clima_df['time_local'] <= row['fecha_fin'])
    periodo = clima_df.loc[mask]
    
    if periodo.empty:
        return pd.Series({'temp_suelo_prom': np.nan, 'lluvia_total': np.nan})
    
    return pd.Series({
        'temp_suelo_prom': periodo['stl1_celsius'].mean(),
        'lluvia_total': periodo['tp_mm'].sum(),
        'viento_max': periodo['wind_speed_10m_ms'].max()
    })

In [9]:

weather_stats = df_joined.apply(lambda row: get_incubation_weather(row, df_joined), axis=1)
df_ml = pd.concat([df_joined, weather_stats], axis=1)

KeyError: 'fecha_inicio'

In [ ]:
df_ml.info()

In [ ]:
df_ml['ESPECIE'].value_counts()

In [ ]:
# Convertir todo a mayúscula inicial para evitar duplicados por formato
df_ml['ESPECIE'] = df_ml['ESPECIE'].str.capitalize()

In [ ]:
# Convertir la columna ESPECIE en columnas numéricas (Dummies)
df_procesado = pd.get_dummies(df_ml, columns=['ESPECIE'], prefix='ESP')

In [ ]:
df_procesado.head()

## Modelos: Random Forest vs XGBoost

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
# Selección de variables (Ejemplo)
features = [
    'ESP_Cm', 'ESP_Cc', 'ESP_Ei', # Las nuevas columnas de especie
    'TAMAÑO DE NIDADA', 
    'temp_suelo_prom', 
    'lluvia_total', 
    'viento_max', 
    'DIAS DE INCUBACION'
]
target = '% DE SOBREVIVENCIA'

In [ ]:
# Limpiar nulos en el target y features
df_procesado = df_procesado.dropna(subset=[target] + features)

In [ ]:
# Limpiar nulos
df_procesado = df_procesado.dropna(subset=[target] + features)

X = df_procesado[features]
y = df_procesado[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# --- Random Forest ---
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

In [ ]:
# --- XGBoost ---
xgb = XGBRegressor(n_estimators=100, learning_rate=0.05)
xgb.fit(X_train, y_train)

In [ ]:
# Ver la importancia de las variables xgboost
importances = pd.Series(xgb.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False).head(10))

In [ ]:
# Ver la importancia de las variables rf
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False).head(10))